# Experiments summary (A, B)

Recorded results of experiments A and B, copied from the saved outputs of `A_apple_heldout.ipynb` and `B_apple_transductive.ipynb`. No training happens here. Provenance flags (`src`) mark numbers that are not in the saved notebook outputs.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# exp, method, FBR, CDAN+E, MT, backbone, SAM, acc, src
#   src: 'nb' = saved notebook output, 'old' = earlier README only, 'const' = hard-coded constant, 'sam?' = SAM unconfirmed
ABL = [
 ('A','Source only',          '-','-','-','ResNet-18','-',    60.40,'const'),
 ('A','CDAN+E',               'Y','Y','-','ResNet-18','ViT-B',89.74,'nb'),
 ('A','CDAN+E',               'Y','Y','-','ResNet-18','ViT-H',94.08,'old'),
 ('A','MT only',              'Y','-','Y','ResNet-18','?',    91.04,'sam?'),
 ('A','CDAN+E + MT',          'Y','Y','Y','ResNet-18','ViT-B',95.81,'nb'),
 ('A','CDAN+E + MT',          'Y','Y','Y','ResNet-18','ViT-H',96.82,'old'),
 ('A','CDAN+E + MT',          'Y','Y','Y','ResNet-50','ViT-H',97.54,'nb'),
 ('B','CDAN+E',               'Y','Y','-','ResNet-50','ViT-H',92.95,'nb'),
 ('B','CDAN+E + MT',          'Y','Y','Y','ResNet-50','ViT-H',98.50,'nb'),
]
abl = pd.DataFrame(ABL, columns=['Exp','Method','FBR','CDAN+E','MT','Backbone','SAM','Acc','src'])
abl

## Table 1 - Ablation
Paired differences (same backbone and SAM):

In [ ]:
def d(a, b): return round(b - a, 2)
pd.Series({
 'A  +MT on CDAN+E (ViT-B)': d(89.74, 95.81),
 'A  +MT on CDAN+E (ViT-H)': d(94.08, 96.82),
 'A  SAM ViT-B -> ViT-H, CDAN+E': d(89.74, 94.08),
 'A  SAM ViT-B -> ViT-H, CDAN+E+MT': d(95.81, 96.82),
 'A  ResNet-18 -> ResNet-50 (CDAN+E+MT, ViT-H)': d(96.82, 97.54),
 'B  +MT on CDAN+E': d(92.95, 98.50),
}, name='delta (points)')

## Table 2 - Comparison with published results

In [ ]:
CMP = pd.DataFrame([
 ('A','apple, held-out 692','ResNet-18','CDAN+E 89.74/94.08; CDAN+E+MT 95.81/96.82','Jeon 2026: CDAN+E w/ FBR 91.1+-4.22, baseline 47.1, Real-to-Real 97.7','Yes, closest'),
 ('B','apple, transductive 1730','ResNet-50','CDAN+E 92.95; CDAN+E+MT 98.50','MSUN 72.31, DANN 50.04, baseline 39.27 (no FBR)','Partly (FBR confound)'),
], columns=['Exp','Pair, protocol','Backbone','Ours','Published','Comparable?'])
CMP

## Charts

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
a = abl[abl.Exp == 'A'].copy()
a['label'] = a.Method + '\n' + a.Backbone + ' ' + a.SAM
ax[0].barh(a.label, a.Acc, color='#4c78a8'); ax[0].invert_yaxis(); ax[0].set_xlim(40, 100)
ax[0].axvline(91.1, ls='--', c='gray'); ax[0].text(91.3, 0, 'Jeon CDAN+E 91.1', fontsize=8)
ax[0].set_title('A - ablation (test acc %)')

b_names = ['Baseline','DANN','MRAN','MSUN','CDAN+E (ours)','CDAN+E+MT (ours)']
b_acc   = [39.27, 50.04, 67.45, 72.31, 92.95, 98.50]
ax[1].bar(b_names, b_acc, color=['#9aa0a6']*4 + ['#4c78a8', '#e45756'])
ax[1].set_ylim(0, 100); ax[1].tick_params(axis='x', rotation=30); ax[1].set_title('B - vs MSUN Table 6 (acc %)')
plt.tight_layout(); plt.show()